# Independent CPU readback of saved group03 compact proof
Six disjoint 20-job shards; no training or Final.


In [ ]:
import hashlib
import json
import pathlib

SOURCE_NOTEBOOK = "hungluongduc/hung-compact-proof-group03-20261028-v1"
SOURCE_VERSION_ID = 363111778
PROOFS = {
    "shard_0008": {
        "bytes": 25086,
        "sha256": "460ee5c8703ca6a35f7d4e1a06e038d93b5d8eecd5c3f8df7ad583ac0fcfbd38",
        "archive_bytes": 74362442,
        "archive_sha256": "a49fc9c6c56b5e6cdcf54bb1af15d05cb511d017f3be3cb4abfb41f17dc151e8",
        "jobset_sha256": "928996ffad7ba48f210d2b69cc23cdf4c402a8d34c9a79be76a58a1591d7ff9d",
    },
    "shard_0009": {
        "bytes": 25098,
        "sha256": "509f30f21455a65579a7a2d8e0083143747989495295d36d5f3272ca45f23f51",
        "archive_bytes": 68786447,
        "archive_sha256": "37f635185da5bf08fd927e0ecacb9efb53f41f2b87f85ad3554f5a92ee53de5e",
        "jobset_sha256": "ac51753e9c13f2e6f0677fda3e1e67174fb8a2080a7a859e02465a0c790278f0",
    },
    "shard_0010": {
        "bytes": 25044,
        "sha256": "6f03412c5386dd65d7c99bf8e7d67e2d9fa3a85b82454749146c48bf725b3a3f",
        "archive_bytes": 66147049,
        "archive_sha256": "c83b3af82e5eedc2c4b68d0bb1d808b10e66fb037708bb5345c38ae3c602b180",
        "jobset_sha256": "9ec71c246ca5334cc5a622cbff9e7b76e6535e447ce0ae6520a8ee9095dc34a3",
    },
    "shard_0011": {
        "bytes": 22791,
        "sha256": "632d6531c2d53644a9a1ab7209f04c2067b9e0e6b76973cf3268b9bb0204309e",
        "archive_bytes": 317485131,
        "archive_sha256": "959dbb76e407ac6c1ff7fce723fdc1510046280b1b473902ede231ff79bcfc6b",
        "jobset_sha256": "7ff773c49768e8f73a3b7245e6b6a178de0f599adccee87234f2c4dfa2299932",
    },
    "shard_0012": {
        "bytes": 21584,
        "sha256": "dcea62992dd8c65824739b69571727ad68db33a97d7022c8fde55c08c747ba9b",
        "archive_bytes": 452751164,
        "archive_sha256": "d0f7f0335745843adbf16b5c05aafd7fb7a4505ba13c92ab649849677d9b92dc",
        "jobset_sha256": "ae3f9b967f922de325bd35c045eeaafddb3647da47c09d2749887c0351102017",
    },
    "shard_0013": {
        "bytes": 21569,
        "sha256": "b43feb6603fd6f09f5202411da643c8fa306b7ae4c999edef3cce5e624b01a36",
        "archive_bytes": 453014419,
        "archive_sha256": "49442d6a54209957e2fc008f3f7891b5160c7a9e23fa7c5eee7c49f46f710b51",
        "jobset_sha256": "d19ce730e5d2bd6d02c545359838569e1b712269689a58f9ca0e1a03d30b18b8",
    },
}
source_root = pathlib.Path("/kaggle/input")
seen_jobs = set()
readbacks = []
for shard_id, pin in PROOFS.items():
    matches = [
        path
        for path in source_root.rglob(f"{shard_id}-compact-proof.json")
        if "hung-compact-proof-group03-20261028-v1" in str(path)
    ]
    assert len(matches) == 1, f"Saved proof missing or ambiguous: {shard_id}"
    raw = matches[0].read_bytes()
    assert len(raw) == pin["bytes"]
    assert hashlib.sha256(raw).hexdigest() == pin["sha256"]
    proof = json.loads(raw)
    assert proof["schema"] == "hung-kaggle-compact-shard-proof-v1"
    assert proof["scope"] == "development_only"
    assert proof["shard_id"] == shard_id
    assert proof["archive_sha256"] == pin["archive_sha256"]
    assert proof["archive_bytes"] == pin["archive_bytes"]
    rows = proof["rows"]
    ids = [row["job_id"] for row in rows]
    assert len(ids) == len(set(ids)) == 20
    assert all(
        row["status"] == "VERIFIED_ARCHIVE_BYTES" and row["shard_id"] == shard_id for row in rows
    )
    assert not seen_jobs.intersection(ids)
    seen_jobs.update(ids)
    jobset_sha = hashlib.sha256(json.dumps(ids, separators=(",", ":")).encode("utf-8")).hexdigest()
    assert jobset_sha == pin["jobset_sha256"]
    readbacks.append(
        {
            "shard_id": shard_id,
            "proof_bytes": len(raw),
            "proof_sha256": hashlib.sha256(raw).hexdigest(),
            "jobset_sha256": jobset_sha,
            "job_ids": ids,
            "jobs": 20,
            "status": "REMOTE_PROOF_READBACK_VERIFIED",
        }
    )
assert len(seen_jobs) == 120
readback = {
    "schema": "hung-kaggle-compact-proof-group-readback-v1",
    "scope": "development_only",
    "source_notebook": SOURCE_NOTEBOOK,
    "claimed_source_version_id": SOURCE_VERSION_ID,
    "readbacks": readbacks,
    "jobs": 120,
    "status": "REMOTE_PROOF_READBACK_VERIFIED",
}
out = pathlib.Path("/kaggle/working/hung-compact-proof-group03-readback-20261028-v1.json")
with out.open("x", encoding="utf-8", newline="\n") as stream:
    json.dump(readback, stream, sort_keys=True, indent=2, allow_nan=False)
    stream.write("\n")
print(
    json.dumps(
        {
            "status": readback["status"],
            "jobs": 120,
            "readbacks": [
                {
                    "shard_id": row["shard_id"],
                    "proof_bytes": row["proof_bytes"],
                    "proof_sha256": row["proof_sha256"],
                    "jobset_sha256": row["jobset_sha256"],
                }
                for row in readbacks
            ],
        },
        sort_keys=True,
    )
)